# Module 2: Data Cleaning and Structuring

Cleans `military_raw_data.csv` (from the Global Firepower scrape) into a Tableau-ready `military_cleaned.csv`.

**Steps:**
1. Clean text: remove `$`, commas, `%`, `+`, and unit-label symbols
2. Convert metrics to numeric formats
3. Drop duplicate columns
4. Standardize column names (e.g. `total_aircraft`, `active_personnel`)
5. Handle missing/null values
6. Save a clean dataset for Tableau input

In [1]:
import re
import numpy as np
import pandas as pd

RAW_FILE = "military_raw_data.csv"
CLEAN_FILE = "military_cleaned.csv"

TEXT_COLUMNS = ["Country", "Continent", "Region", "Alliance"]

## Step 1: Load the raw data

In [2]:
df = pd.read_csv(RAW_FILE)
print(df.shape)
df.head()

(145, 80)


,Country,Active_Personnel,Total_Aircraft,Aircraft_Carriers,Airports,Total_Area,Armored_Vehicles,Defense_Budget,Destroyers,Fighters,...,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km,Continent,Region,Alliance
0,Afghanistan,75000,5,0,68,652230,3902,145000000,0,0,...,"767,000 \t\t\t\t\t\t\n\...","503,000 \t\t\t\t\t\t\n\...","66,000,000 \t\t\t\t\t\t...","652,230 \t\t\t\t\t\t\n\...",0 \t\t\t\t\t\t\n\t\t\t\...,"5,987 \t\t\t\t\t\t\n\t\...","1,200 \t\t\t\t\t\t\n\t\...",Asia,Southern Asia,NaN
1,Albania,7500,20,0,3,28748,1492,720037190,0,0,...,"473,000 \t\t\t\t\t\t\n\...","255,000 \t\t\t\t\t\t\n\...","522,000,000 \t\t\t\t\t\...","28,748 \t\t\t\t\t\t\n\t...",362 \t\t\t\t\t\t\n\t\t\...,691 \t\t\t\t\t\t\n\t\t\...,41 \t\t\t\t\t\t\n\t\t\t...,Europe,Southern Europe,NATO
2,Algeria,130000,620,0,95,2381740,24920,25000000000,0,111,...,0 \t\t\t\t\t\t\n\t\t\t\...,"3,000 \t\t\t\t\t\t\n\t\...","233,000,000 \t\t\t\t\t\...","2,381,740 \t\t\t\t\t\t\...",998 \t\t\t\t\t\t\n\t\t\...,"6,734 \t\t\t\t\t\t\n\t\...",0 \t\t\t\t\t\t\n\t\t\t\...,Africa,Northern Africa,NaN
3,Angola,107000,278,0,107,1246700,2258,31200000000,0,67,...,0 \t\t\t\t\t\t\n\t\t\t\...,0 \t\t\t\t\t\t\n\t\t\t\...,0 \t\t\t\t\t\t\n\t\t\t\...,"1,246,700 \t\t\t\t\t\t\...","1,600 \t\t\t\t\t\t\n\t\...","5,369 \t\t\t\t\t\t\n\t\...","1,300 \t\t\t\t\t\t\n\t\...",Africa,Sub-Saharan Africa,NaN
4,Argentina,108000,241,0,764,2780400,22432,993919790,3,23,...,"869,000 \t\t\t\t\t\t\n\...","2,534,000 \t\t\t\t\t\t\...","799,999,000 \t\t\t\t\t\...","2,780,400 \t\t\t\t\t\t\...","4,989 \t\t\t\t\t\t\n\t\...","11,968 \t\t\t\t\t\t\n\t...","11,000 \t\t\t\t\t\t\n\t...",Americas,Latin America and the Caribbean,NaN


## Step 2: Clean text -> numeric
Removes `$`, commas, `%`, `+`, and unit labels (`km`, `bbl`, `Cu.M`, `mt`, ...) that were wrapped onto a new line during scraping.

In [3]:
def clean_numeric_value(raw_value):
    """Turn a messy scraped value into a plain float (or NaN)."""
    if pd.isna(raw_value):
        return np.nan

    text = str(raw_value)
    # Unit labels were appended after a line break during scraping
    text = text.split("\n")[0]
    # Remove currency symbols, thousands separators, percent and plus signs
    text = re.sub(r"[\$,%+]", "", text)
    text = text.strip()

    if text == "" or text.lower() == "nan":
        return np.nan
    try:
        return float(text)
    except ValueError:
        return np.nan

numeric_columns = [c for c in df.columns if c not in TEXT_COLUMNS]
for column in numeric_columns:
    df[column] = df[column].apply(clean_numeric_value)

print(f"Cleaned {len(numeric_columns)} numeric columns")
df.head()

Cleaned 76 numeric columns


,Country,Active_Personnel,Total_Aircraft,Aircraft_Carriers,Airports,Total_Area,Armored_Vehicles,Defense_Budget,Destroyers,Fighters,...,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km,Continent,Region,Alliance
0,Afghanistan,75000.0,5.0,0.0,68.0,652230.0,3902.0,1.450000e+08,0.0,0.0,...,767000.0,503000.0,66000000.0,652230.0,0.0,5987.0,1200.0,Asia,Southern Asia,NaN
1,Albania,7500.0,20.0,0.0,3.0,28748.0,1492.0,7.200372e+08,0.0,0.0,...,473000.0,255000.0,522000000.0,28748.0,362.0,691.0,41.0,Europe,Southern Europe,NATO
2,Algeria,130000.0,620.0,0.0,95.0,2381740.0,24920.0,2.500000e+10,0.0,111.0,...,0.0,3000.0,233000000.0,2381740.0,998.0,6734.0,0.0,Africa,Northern Africa,NaN
3,Angola,107000.0,278.0,0.0,107.0,1246700.0,2258.0,3.120000e+10,0.0,67.0,...,0.0,0.0,0.0,1246700.0,1600.0,5369.0,1300.0,Africa,Sub-Saharan Africa,NaN
4,Argentina,108000.0,241.0,0.0,764.0,2780400.0,22432.0,9.939198e+08,3.0,23.0,...,869000.0,2534000.0,799999000.0,2780400.0,4989.0,11968.0,11000.0,Americas,Latin America and the Caribbean,NaN


## Step 3: Drop duplicate columns
The raw file contains an early, already-clean subset of columns (e.g. `Active_Personnel`) that duplicate a later, messier scraped column (e.g. `active_personnel`) for every row.

In [4]:
def drop_duplicate_columns(dataframe):
    seen = {}
    columns_to_drop = []
    for column in dataframe.columns:
        fingerprint = tuple(dataframe[column].fillna("NA").astype(str))
        if fingerprint in seen:
            columns_to_drop.append(column)
        else:
            seen[fingerprint] = column
    return dataframe.drop(columns=columns_to_drop), columns_to_drop

df, dropped_columns = drop_duplicate_columns(df)
print(f"Dropped {len(dropped_columns)} duplicate columns:")
dropped_columns

Dropped 21 duplicate columns:


['total_population',
 'active_personnel',
 'reserve_personnel',
 'total_military_aircraft',
 'fighter_aircraft',
 'total_military_helicopters',
 'tanks',
 'armored_fighting_vehicles',
 'rocket_projectors',
 'total_naval_fleet',
 'aircraft_carriers',
 'submarines',
 'destroyers',
 'frigates',
 'defense_budget_usd',
 'purchasing_power_parity_usd',
 'total_serviceable_airports',
 'major_ports_and_terminals',
 'railway_coverage_km',
 'roadway_coverage_km',
 'total_land_area_sq_km']

## Step 4: Standardize column names to lower_snake_case

In [5]:
def to_snake_case(column_name):
    name = column_name.strip()
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name.lower()

df.columns = [to_snake_case(c) for c in df.columns]
df.columns.tolist()

['country',
 'active_personnel',
 'total_aircraft',
 'aircraft_carriers',
 'airports',
 'total_area',
 'armored_vehicles',
 'defense_budget',
 'destroyers',
 'fighters',
 'fleet_strength',
 'frigates',
 'gdp',
 'helicopters',
 'population',
 'ports',
 'railways',
 'reserve_personnel',
 'roadways',
 'rocket_projectors',
 'submarines',
 'tanks',
 'power_index',
 'total_military_manpower',
 'fit_for_service',
 'population_reaching_military_age_annually',
 'paramilitary',
 'attack_aircraft',
 'transport_aircraft',
 'trainer_aircraft',
 'special_mission_aircraft',
 'tanker_aircraft',
 'attack_helicopters',
 'self_propelled_artillery',
 'towed_artillery',
 'total_naval_fleet_tonnage_mt',
 'helicopter_carriers',
 'corvettes',
 'coastal_patrol_craft',
 'mine_warfare_craft',
 'external_debt_usd',
 'foreign_exchange_and_gold_reserves_usd',
 'labour_force',
 'total_merchant_marine_fleet',
 'oil_production_bbl',
 'oil_consumption_bbl',
 'proven_oil_reserves_bbl',
 'natural_gas_production_cum',
 'n

## Step 5: Handle missing / null values
- `alliance`: blank means the country isn't part of a listed alliance -> `"Non-aligned"`
- `continent` / `region`: fill stray blanks with `"Unknown"`
- remaining numeric columns: missing means "not reported" -> `0`

In [6]:
if "alliance" in df.columns:
    df["alliance"] = df["alliance"].fillna("Non-aligned")

for column in ["continent", "region"]:
    if column in df.columns:
        df[column] = df[column].fillna("Unknown")

numeric_cols_final = df.select_dtypes(include=[np.number]).columns
df[numeric_cols_final] = df[numeric_cols_final].fillna(0)

missing_pct = df.isnull().mean().mean() * 100
print(f"Missing values after cleaning: {missing_pct:.2f}%")

Missing values after cleaning: 0.00%


## Step 6: Save the cleaned dataset

In [7]:
df.to_csv(CLEAN_FILE, index=False)
print(f"Saved -> {CLEAN_FILE}")
print(f"Final shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Saved -> military_cleaned.csv
Final shape: 145 rows, 59 columns


,country,active_personnel,total_aircraft,aircraft_carriers,airports,total_area,armored_vehicles,defense_budget,destroyers,fighters,...,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,coastline_coverage_km,border_coverage_km,waterway_coverage_km,continent,region,alliance
0,Afghanistan,75000.0,5.0,0.0,68.0,652230.0,3902.0,1.450000e+08,0.0,0.0,...,4.955400e+10,767000.0,503000.0,66000000.0,0.0,5987.0,1200.0,Asia,Southern Asia,Non-aligned
1,Albania,7500.0,20.0,0.0,3.0,28748.0,1492.0,7.200372e+08,0.0,0.0,...,5.692000e+09,473000.0,255000.0,522000000.0,362.0,691.0,41.0,Europe,Southern Europe,NATO
2,Algeria,130000.0,620.0,0.0,95.0,2381740.0,24920.0,2.500000e+10,0.0,111.0,...,4.504000e+12,0.0,3000.0,233000000.0,998.0,6734.0,0.0,Africa,Northern Africa,Non-aligned
3,Angola,107000.0,278.0,0.0,107.0,1246700.0,2258.0,3.120000e+10,0.0,67.0,...,3.430020e+11,0.0,0.0,0.0,1600.0,5369.0,1300.0,Africa,Sub-Saharan Africa,Non-aligned
4,Argentina,108000.0,241.0,0.0,764.0,2780400.0,22432.0,9.939198e+08,3.0,23.0,...,3.964640e+11,869000.0,2534000.0,799999000.0,4989.0,11968.0,11000.0,Americas,Latin America and the Caribbean,Non-aligned
